In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle

In [20]:
df = pd.read_csv("merged_dataset.csv")

In [21]:
print(df["Crop"].head())

0        Arecanut
1    Black pepper
2       Cashewnut
3        Coconut 
4         Tapioca
Name: Crop, dtype: object


In [3]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4426 entries, 0 to 4425
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Crop             4426 non-null   object 
 1   Crop_Year        4426 non-null   int64  
 2   Season           4426 non-null   object 
 3   State            4426 non-null   object 
 4   Area             4426 non-null   float64
 5   Production       4426 non-null   int64  
 6   Annual_Rainfall  4426 non-null   float64
 7   Fertilizer       4426 non-null   float64
 8   Pesticide        4426 non-null   float64
 9   Yield            4426 non-null   float64
dtypes: float64(5), int64(2), object(3)
memory usage: 345.9+ KB


In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Crop"] = le.fit_transform(df["Crop"])
df["Season"] = le.fit_transform(df["Season"])
df["State"] = le.fit_transform(df["State"])

In [5]:
df["Yield_Category"] = pd.qcut(
    df["Yield"],
    q=5,   # 5 equal groups
    labels=False,
    duplicates="drop"
)

In [6]:
print(df["Yield_Category"].value_counts())

Yield_Category
0    886
2    885
1    885
4    885
3    885
Name: count, dtype: int64


In [7]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(df, df["Yield_Category"]):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]

In [8]:
X_train = strat_train_set.drop(["Yield", "Yield_Category"], axis=1)
y_train = strat_train_set["Yield"]

X_test = strat_test_set.drop(["Yield", "Yield_Category"], axis=1)
y_test = strat_test_set["Yield"]

In [9]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [10]:
y_pred = model.predict(X_test)

In [11]:
print("R2 Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R2 Score: 0.855680434621511
MAE: 18.63075179637638


In [12]:
pickle.dump(model, open("models/crop_model.pkl", "wb"))

In [13]:
df.drop("Yield_Category", axis=1, inplace=True)

In [18]:
df

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,0,1997,4,5,76145.0,93995,3252.4,7246719.65,23604.95,1.147857
1,5,1997,4,5,173855.0,55520,3252.4,16545780.35,53895.05,0.240714
2,7,1997,4,5,96073.0,74142,3252.4,9143267.41,29782.63,0.589286
3,9,1997,4,5,884344.0,5210000000,3252.4,84163018.48,274146.64,5376.054286
4,48,1997,4,5,132875.0,2841819,3252.4,12645713.75,41191.25,22.803571
...,...,...,...,...,...,...,...,...,...,...
4421,43,1997,1,0,18467.0,18032,2274.9,1757504.39,5724.77,0.983077
4422,44,1997,1,0,2345.0,3075,2274.9,223173.65,726.95,1.493077
4423,45,1997,2,0,681.0,15806,2274.9,64810.77,211.11,20.801111
4424,46,1997,1,0,448.0,220,2274.9,42636.16,138.88,0.491538


In [22]:
df = pd.read_csv("merged_dataset.csv")

In [26]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(df["Crop"])

clean_mapping = {k: int(v) for k, v in mapping.items()}

print(clean_mapping)

{'Arecanut': 0, 'Arhar/Tur': 1, 'Bajra': 2, 'Banana': 3, 'Barley': 4, 'Black pepper': 5, 'Cardamom': 6, 'Cashewnut': 7, 'Castor seed': 8, 'Coconut ': 9, 'Coriander': 10, 'Cotton(lint)': 11, 'Cowpea(Lobia)': 12, 'Dry chillies': 13, 'Garlic': 14, 'Ginger': 15, 'Gram': 16, 'Groundnut': 17, 'Guar seed': 18, 'Horse-gram': 19, 'Jowar': 20, 'Jute': 21, 'Khesari': 22, 'Linseed': 23, 'Maize': 24, 'Masoor': 25, 'Mesta': 26, 'Moong(Green Gram)': 27, 'Moth': 28, 'Niger seed': 29, 'Oilseeds total': 30, 'Onion': 31, 'Other  Rabi pulses': 32, 'Other Cereals': 33, 'Other Kharif pulses': 34, 'Peas & beans (Pulses)': 35, 'Potato': 36, 'Ragi': 37, 'Rapeseed &Mustard': 38, 'Rice': 39, 'Safflower': 40, 'Sannhamp': 41, 'Sesamum': 42, 'Small millets': 43, 'Soyabean': 44, 'Sugarcane': 45, 'Sunflower': 46, 'Sweet potato': 47, 'Tapioca': 48, 'Tobacco': 49, 'Turmeric': 50, 'Urad': 51, 'Wheat': 52, 'other oilseeds': 53}


In [28]:
import json

with open("models/crop_mapping.json", "w") as f:
    json.dump(clean_mapping, f)

In [31]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import json

# STEP 1: Load original dataset
df = pd.read_csv("merged_dataset.csv")

# =========================
# 🌦️ SEASON MAPPING
# =========================
le_season = LabelEncoder()
le_season.fit(df["Season"])

season_mapping = dict(zip(le_season.classes_, le_season.transform(le_season.classes_)))

# Convert np.int64 → int
season_mapping = {k: int(v) for k, v in season_mapping.items()}

print("Season Mapping:")
print(season_mapping)

# Save
with open("models/season_mapping.json", "w") as f:
    json.dump(season_mapping, f)


# =========================
# 📍 STATE MAPPING
# =========================
le_state = LabelEncoder()
le_state.fit(df["State"])

state_mapping = dict(zip(le_state.classes_, le_state.transform(le_state.classes_)))

# Convert np.int64 → int
state_mapping = {k: int(v) for k, v in state_mapping.items()}

print("\nState Mapping:")
print(state_mapping)

# Save
with open("models/state_mapping.json", "w") as f:
    json.dump(state_mapping, f)

Season Mapping:
{'Autumn     ': 0, 'Kharif     ': 1, 'Rabi       ': 2, 'Summer     ': 3, 'Whole Year ': 4, 'Winter     ': 5}

State Mapping:
{'ARUNACHAL PRADESH': 0, 'BIHAR': 1, 'CHHATTISGARH': 2, 'HIMACHAL PRADESH': 3, 'JHARKHAND': 4, 'KERALA': 5, 'PUNJAB': 6, 'TAMIL NADU': 7, 'TELANGANA': 8, 'UTTARAKHAND': 9}
